In [4]:
from neo4j import GraphDatabase
import logging
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [5]:
NEO4J_CONFIG = {
    "uri": "bolt://neo4j_nosql_lab:7687", 
    "user": "neo4j",
    "password": "test1234"
}

In [9]:
driver = GraphDatabase.driver(NEO4J_CONFIG["uri"], auth=(NEO4J_CONFIG["user"], NEO4J_CONFIG["password"]))

query_cypher = """
    MATCH (u1:User)-[t1:TRANSFER]->()
    MATCH (u1)-[:USES]->(d:Device)<-[:USES]-(u2:User)
    MATCH (u2)-[t2:TRANSFER]->()
    WHERE elementId(u1) < elementId(u2)
        AND duration.inDays(datetime(t1.timestamp), datetime(t2.timestamp)).days <= 3
    RETURN DISTINCT u1.name AS Uzytkownik_1, u2.name AS Uzytkownik_2, d.device_id AS Wspolne_Urzadzenie
"""

with driver.session() as session:
    result = session.run(query_cypher)
    print("--- Wyszukiwanie użytkowników korzystających z tego samego urządzenia w ciągu ostatnich 3 dni. ---")

    found = False
    for record in result:
        found = True
        print(f" Użytkownicy {record['Uzytkownik_1']} oraz {record['Uzytkownik_2']} korzystali z tego samego urządzenia: {record['Wspolne_Urzadzenie']}")
        
    if not found:
        print("Brak użytkowników korzystających z tego samego urządzenia w ciągu ostatnich 3 dni.")

driver.close()

--- Wyszukiwanie użytkowników korzystających z tego samego urządzenia w ciągu ostatnich 3 dni. ---
 Użytkownicy user11 oraz user9 korzystali z tego samego urządzenia: shared1
 Użytkownicy user9 oraz user13 korzystali z tego samego urządzenia: shared1
 Użytkownicy user11 oraz user13 korzystali z tego samego urządzenia: shared1
 Użytkownicy user12 oraz user1 korzystali z tego samego urządzenia: devY
 Użytkownicy user1 oraz user10 korzystali z tego samego urządzenia: devY
 Użytkownicy user12 oraz user10 korzystali z tego samego urządzenia: devY
 Użytkownicy user11 oraz user2 korzystali z tego samego urządzenia: devZ
 Użytkownicy user2 oraz user14 korzystali z tego samego urządzenia: devZ
 Użytkownicy user11 oraz user14 korzystali z tego samego urządzenia: devZ
 Użytkownicy user17 oraz user15 korzystali z tego samego urządzenia: shared2
 Użytkownicy user12 oraz user1 korzystali z tego samego urządzenia: devA
 Użytkownicy user12 oraz user16 korzystali z tego samego urządzenia: devA
 Użytkow